# Reproducing Ocagli et al. (2024) from Figshare item 26877211

**Paper.** Ocagli H, Lanera C, Franzoi M, Monachesi C, Zgheib R, Belluco S, Dorne JLCM, Gregori D,
Baldi I (2024). Unveiling insights from the Joint FAO/WHO Expert Committee on Food Additives (JECFA)
portal. *Scientific Data*. [doi:10.1038/s41597-024-04294-w](https://doi.org/10.1038/s41597-024-04294-w).
The claims file was written from the version of record in Europe PMC (PMC11682307, full-text XML),
a Data Descriptor with no supplement.

**Dataset.** [Figshare 26877211](https://springernature.figshare.com/articles/dataset/Unveiling_Insights_from_the_Joint_FAO_WHO_Expert_Committee_on_Food_Additives_JECFA_Portal/26877211/1),
version 1, [doi:10.6084/m9.figshare.26877211.v1](https://doi.org/10.6084/m9.figshare.26877211.v1).
The paper cites the item without a version (10.6084/m9.figshare.26877211), which resolves to the
latest; the item has one published version, 1 (posted 2024-12-29), so that is the version pinned.
It holds two files, read here by streaming into memory: `jecfa.rds` (file id 48922360, MD5
5bfd89459ec77a152e5ee1f55e124f3d) and `JECFA.csv` (file id 48922363, MD5
dee04993aaa56ca1f3844ded5c472764). The MD5 of the bytes read is checked against the archive's
`computed_md5` in the data check.

**Code.** Method details the claims file takes from the paper's code refer to
https://github.com/UBESP-DCTV/jecfa at commit 2d8a5340d5322cbae28eb3095d948313932f6c7e; the FAS
derivations in `jecfa.py` follow `R/process_df.R` and `R/add_metadata.R` there. Nothing here is
random, so no seed is needed.

| Claim | Tested |
|---|---|
| data-records-rows (link): 6552 rows with the fields of Table 1 | tested |
| unique-pages: 6549 unique pages after removing a duplicate and a test page | tested |
| table1-availability: share of pages with each field | tested |
| table2-evaluation-year: evaluations by year bin | tested |
| table2-functional-class: substances by functional class | tested |
| unknowns-name-only: unknowns have only a JECFA name | tested |
| fas-codes: FAS 40 to 52 (even) cover 50% of records | tested |
| validation-failures: discrepancies in the 700-page manual check | blocked (data_missing) |
| portal-search-count: portal count 6547 in line with pages with a JECFA name | tested (dataset's part) |

In [ ]:
import time

import pandas as pd

import jecfa

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)

started = time.time()
data = jecfa.load()
df = data["rds"]
print(f"Figshare item {jecfa.ARTICLE_ID} version {jecfa.VERSION} ({jecfa.DOI}), "
      f"read in {time.time() - started:.1f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
meta = data["meta"]
print("Title:", meta["title"])
print("Published:", meta["published_date"], " version:", meta["version"], " DOI:", meta["doi"])
print("Related:", [(r["relation"], r["identifier"]) for r in meta["related_materials"]])
print("\nFiles (name, id, size in bytes, MD5 of the bytes read, archive computed_md5, equal):")
for f in meta["files"]:
    got, expected = data["md5"][f["name"]]
    print(f"  {f['name']:10s} {f['id']}  {f['size']:>8d}  {got}  {expected}  {got == expected}")
versions = pd.read_json(jecfa.API_URL.rsplit("/", 1)[0])
print("\nPublished versions of the item:", versions["version"].tolist())
print("\nThe item's description, which documents each field:\n")
print(meta["description"])

The item has one published version, and the bytes read match the archive's MD5 for both files.
The description documents the 17 fields. It calls FAS a "FAO/WHO Food Additive Specification
identifier"; the paper (Table 1) and the code (`R/process_df.R`) make it the number of the Food
Additives Series monograph, taken from the Tox monograph. The check below shows which the data
hold.

In [ ]:
csv = data["csv"]
print("jecfa.rds:", df.shape, " JECFA.csv:", csv.shape)
print("Same column names in the same order:", list(df.columns) == list(csv.columns))
same = (df.fillna("<NA>").to_numpy() == csv.fillna("<NA>").to_numpy())
print("Cells that differ between the two files:", int((~same).sum()))
print("\nColumns of jecfa.rds, next to the field of Table 1 and the name in R/process_df.R:")
cols = pd.DataFrame({"table1_field": list(jecfa.TABLE1_FIELDS),
                     "deposit_column": list(jecfa.TABLE1_FIELDS.values())})
code_names = {c.lower().replace(".", "_"): c for c in jecfa.CODE_COLUMNS}
cols["process_df_name"] = cols.deposit_column.map(code_names)
cols["in_deposit"] = cols.deposit_column.isin(df.columns)
print(cols.to_string(index=False))
print("Deposit columns not in Table 1:", sorted(set(df.columns) - set(cols.deposit_column)))

The two files hold the same table: the CSV is a copy of the RDS, cell for cell. The deposit's
column names are those of `R/process_df.R` in lower case with "." replaced by "_", and its columns
are in the order that function selects them. The columns that `R/add_metadata.R` adds (type, host,
Tox_monograph_abbr, Report_clean, ref_id) are not in the deposit, so the deposited table is the
output of `process_df.R`.

In [ ]:
present = df.apply(jecfa.present)
summary = pd.DataFrame({
    "NA": df.isna().sum(),
    "empty_string": (df.notna() & (df.fillna("").apply(lambda s: s.str.strip()) == "")).sum(),
    "present": present.sum(),
    "distinct": df.nunique(),
    "example": [df[c].dropna().iloc[0][:60] for c in df.columns],
})
print(summary.to_string())

Every column is text. Missing values are NA, except in `fas`, which also holds 752 empty strings.
`jecfa_name` and `url` are present in every row.

In [ ]:
pid = jecfa.page_id(df)
print("URLs of the form", jecfa.URL_PREFIX + "<n>:", int(pid.notna().sum()), "of", len(df))
print("Page numbers from", int(pid.min()), "to", int(pid.max()), "; duplicated URLs:", int(df.url.duplicated().sum()),
      "; rows sorted by page number:", bool(pid.is_monotonic_increasing))
missing_ids = sorted(set(range(1, int(pid.max()) + 1)) - set(pid.astype(int)))
print(f"Page numbers between 1 and {int(pid.max())} with no row: {len(missing_ids)}:", missing_ids)

year = pd.to_numeric(df.evaluation_year, errors="coerce")
print("\nevaluation_year: all present values are integers:", bool(year[df.evaluation_year.notna()].notna().all()),
      "; range", int(year.min()), "to", int(year.max()))
for c in ["jecfa_number", "fema_number", "coe_number"]:
    s = df[c].dropna()
    odd = s[~s.str.fullmatch(r"\d+")]
    print(f"{c}: {len(s)} present, {len(odd)} not a plain integer, e.g. {odd.head(4).tolist()}")
cas = df.cas_number.dropna()
plain = cas[cas.str.fullmatch(r"\d{2,7}-\d{2}-\d")]
print(f"cas_number: {len(cas)} present, {len(plain)} of the plain form NNNNNNN-NN-N, of which "
      f"{int(plain.map(jecfa.cas_check_digit_ok).sum())} have a valid check digit; "
      f"{len(cas) - len(plain)} carry more than one number or a note, e.g. {cas[~cas.isin(plain)].head(3).tolist()}")
print("Plain CAS numbers with a wrong check digit:", plain[~plain.map(jecfa.cas_check_digit_ok)].tolist())
print("\nValues starting with the characters '\">' (Usage Notes): report",
      int(df.report.str.startswith('">').sum()), ", tox_monograph", int(df.tox_monograph.str.startswith('">').sum()),
      ", specification", int(df.specification.str.startswith('">').sum()))

The URLs are the portal pages of Table 1, one row per page, with page numbers from 1 to 6593,
inside the 1 to 7000 of the Methods; 41 numbers in that range have no row. No URL is repeated, so
the claims file's two units, U1 (every row) and U2 (one row per distinct URL), are the same here,
and each result below holds for both. Years run from 1958 to 2023, as in Table 2. The identifiers
are mostly plain numbers. A few plain CAS numbers fail the registry's check digit (printed above),
and some report, Tox monograph and specification values start with the stray characters '">'
that the Usage Notes describe.

In [ ]:
fas_rederived = jecfa.fas_process_df(df.tox_monograph)
print("fas equals the derivation of R/process_df.R from tox_monograph in every row:",
      bool((fas_rederived.fillna("<NA>") == df.fas.fillna("<NA>")).all()))
empty_fas = df.fas.fillna("x").str.strip() == ""
print("\nTox monographs of the rows with an empty fas, by the code's abbreviation (R/extract_jecfa_tox_abbr.R):")
print(df.tox_monograph[empty_fas].map(jecfa.tox_abbr).value_counts().to_string())
print("Of these, Tox monographs starting with '\">':", int(df.tox_monograph[empty_fas].str.startswith('">').sum()))
print("\nDistinct fas values:", sorted(df.fas.dropna().unique(), key=lambda s: (len(s), s)))

The `fas` column is the derivation of `R/process_df.R` from the Tox monograph, in every row. It is
the Food Additives Series number, not a specification identifier as the item's description says.
Of the 752 empty strings, 355 are FAS monographs that the "^FAS " pattern misses (339 FAS 60,
10 FAS 74, 2 FAS 27, and 4 written "FAS79"); 349 of the 752 Tox monographs start with '">', which
the pattern does not allow. 277 are
monographs marked "NOT PREPARED", and 120 are not a FAS (such as a TRS reference). Some values keep
the session ("77 JECFA 86") or a page ("67.29"), and one is "617".

In [ ]:
cls = jecfa.leading_class(df.functional_class)
print("Functional class strings, by leading class (the rest of the string is the sub-class):")
print(cls.value_counts(dropna=False).to_string())
print("Strings that start with none of the four class names:",
      df.functional_class[cls.isna() & df.functional_class.notna()].tolist())
print("Spellings of the flavouring class:",
      df.functional_class.str.extract(r"^(Flavou?ring Agent)")[0].value_counts().to_dict())
print("\nCOE number present, by class:")
print(pd.crosstab(cls.fillna("unknown"), present.coe_number).to_string())
print("\nFEMA number present, by class:")
print(pd.crosstab(cls.fillna("unknown"), present.fema_number).to_string())

Every functional class string starts with one of the four classes of Table 1; the data spell
"Flavouring Agent", the paper "Flavoring Agent". Table 1 describes the COE number as assigned by the
Codex Alimentarius Commission, the item's description as the Council of Europe number. COE numbers
are present for 3022 flavouring agents and 9 food additives and no other class, which is consistent
with the Council of Europe numbering of flavourings, not with Codex.

In [ ]:
by_url = df.drop(columns="url")
print("Rows identical to another row in every field but the URL:", int(by_url.duplicated(keep=False).sum()))
dup_names = df.jecfa_name[df.jecfa_name.duplicated(keep=False)]
print("JECFA names used by more than one row:", dup_names.unique().tolist(), "at pages",
      pid[dup_names.index].astype(int).tolist())
print("Rows named TEST:", pid[df.jecfa_name.str.strip().str.upper() == "TEST"].astype(int).tolist())
print("\nFields present in each of these rows:")
show = df.index[pid.isin([5883, 6316, 6319])]
print(present.loc[show].assign(page=pid[show].astype(int)).set_index("page").T.to_string())
print(f"\nData check done in {time.time() - started:.1f} s")

No two rows are copies apart from their URL. The chemical of page 6316 has a second page, 6319:
page 6316 carries only its name, page 6319 the full record. Page 5883 is named TEST and carries
nothing else. These are the duplicate and the test page of the Methods, and both are in the
deposit, which therefore has the 6552 scraped rows, not the 6549 cleaned pages.

The table has no time series, so there is no clock to check against a trial table. Each row is
described by its URL, which ties it to one portal page; the portal has changed since May 2024, so
the rows cannot be checked against the pages as they were scraped.

## data-records-rows (link)

> The deposited dataframe is the JECFA index of May 2024, with 6552 rows and the fields of Table 1.

Criterion: 6552 rows within 3, and a column for each of the 17 fields of Table 1 (by the code's
names in `process_df.R`, or an evident equivalent); JECFA.csv compared with jecfa.rds.

In [ ]:
n_rows = len(df)
fields_found = cols.in_deposit.sum()
print(f"Rows of jecfa.rds: {n_rows} (expected 6552, tolerance 3; within: {abs(n_rows - 6552) <= 3})")
print(f"Table 1 fields with a column: {fields_found} of {len(cols)}; missing: {cols.table1_field[~cols.in_deposit].tolist()}")
print(f"JECFA.csv: {csv.shape[0]} rows and {csv.shape[1]} columns; cells that differ from jecfa.rds: {int((~same).sum())}")

**Result: reproduced.** jecfa.rds has 6552 rows, and all 17 fields of Table 1 have a column, named
as in `process_df.R` in lower case. JECFA.csv holds the same 6552 rows and 17 columns with no
differing cell.

## unique-pages

> The 6552 scraped entries, cleaned of the duplicates of page 6316 and of test pages, give 6549
> unique pages.

Criterion: under one of three readings of the cleaning, 6549 within 2, and page 6316's chemical
appears more than once. (D1) rows with distinct URL; (D2) rows with distinct JECFA name, less the
row of page 5883; (D3) the rows less page 5883 and less all but one row with the JECFA name of
page 6316.

In [ ]:
name_6316 = df.jecfa_name[pid == 6316].iloc[0]
d1 = df.url.nunique()
d2 = df.loc[pid != 5883, "jecfa_name"].nunique()
d3 = len(df) - int((pid == 5883).sum()) - (int((df.jecfa_name == name_6316).sum()) - 1)
readings = pd.DataFrame({"reading": ["D1", "D2", "D3"], "unique_pages": [d1, d2, d3]})
readings["within_2_of_6549"] = (readings.unique_pages - 6549).abs() <= 2
print(readings.to_string(index=False))
print(f"Rows with page 6316's chemical ({name_6316}): {int((df.jecfa_name == name_6316).sum())}, "
      f"at pages {pid[df.jecfa_name == name_6316].astype(int).tolist()}")
print("Variant, removing both rows of that chemical and page 5883:",
      len(df) - int((pid == 5883).sum()) - int((df.jecfa_name == name_6316).sum()))

**Result: reproduced.** Readings D2 and D3 give 6550 unique pages, one more than the paper's 6549
and within the tolerance of 2; D1 gives 6552, as no URL repeats. Page 6316's chemical appears
twice, at pages 6316 and 6319. The paper's 6549 is reached only by removing both rows of that
chemical with the test page.

## table1-availability

> The share of pages carrying each Table 1 field is 100% for URL and JECFA name, 82% chemical names,
> 79% synonyms, 35% CAS number, 71% JECFA number, 65% FEMA number, 46% COE number, 98% functional
> class, 98% evaluation year, 97% report and its link, 93% Tox monograph and its link, 92%
> specification and its link, and 93% FAS.

Criterion: the share of rows where each field is present (reading E), under U1 or U2 (the same
here); at least 15 of 17 within tolerance under one reading for reproduced, 10 to 14 for partially
reproduced. FAS is the column the code derives (`process_df.R`, `add_metadata.R`). The deposit
holds the `process_df.R` column; both are run: (A) the deposited `fas`, (B) `fas` after the step of
`add_metadata.R`. Denominators 6552 rows and the 6550 rows left after cleaning (reading D2 of
unique-pages) are both shown.

In [ ]:
expected = pd.Series({"URL": 100, "JECFA name": 100, "Chemical names": 82, "Synonyms": 79, "CAS number": 35,
                      "JECFA number": 71, "FEMA number": 65, "COE number": 46, "Functional class": 98,
                      "Evaluation year": 98, "Report": 97, "Report sourcelink": 97, "Tox monograph": 93,
                      "Tox monograph sourcelink": 93, "Specification": 92, "Specification sourcelink": 92,
                      "FAS": 93}, dtype=float)
fas_b = jecfa.fas_add_metadata(df.fas, df.tox_monograph)
cleaned = (pid != 5883) & ~((df.jecfa_name == name_6316) & df.jecfa_name.duplicated(keep="last"))
table = pd.DataFrame({"expected": expected, "tolerance": expected * 0.10})
for label, rows in [("all_6552", pd.Series(True, index=df.index)), ("cleaned_6550", cleaned)]:
    pa = present.loc[rows].mean() * 100
    table[f"A_{label}"] = [pa[jecfa.TABLE1_FIELDS[f]] for f in table.index]
    table[f"B_{label}"] = table[f"A_{label}"]
    table.loc["FAS", f"B_{label}"] = jecfa.present(fas_b[rows]).mean() * 100
table = table.round(2)
print(f"Rows used: all {len(df)}; cleaned {int(cleaned.sum())}")
print(table.to_string())
for c in [c for c in table.columns if c[:2] in ("A_", "B_")]:
    within = ((table[c] - table.expected).abs() <= table.tolerance + 1e-9)
    exact = (table[c].round(0) == table.expected)
    print(f"{c}: within tolerance {int(within.sum())} of 17 (outside: {table.index[~within].tolist()}); "
          f"equal to Table 1 when rounded to whole numbers {int(exact.sum())} of 17")
print("\nVariant: FAS counted as present wherever it is not NA (the 752 empty strings counted):",
      round(df.fas.notna().mean() * 100, 2), "%")

**Result: reproduced.** Over all 6552 rows, 16 of 17 shares are within tolerance under reading A
(the deposited FAS column, 81.38% against 93%) and all 17 under reading B (FAS after
`add_metadata.R`, 86.74%); removing the duplicate and the test page changes no share by more than
0.05 points. Rounded to whole numbers, 12 of 17 shares equal Table 1. Under reading A the FAS
share is 11.6 points below Table 1; counting the 752 empty strings as present gives 92.86%, which
suggests that Table 1 counted every non-NA value.

## table2-evaluation-year

> By evaluation year, 2357 evaluations fall in 1958 to 2000 (37%), 3099 in 2000 to 2010 (48%), 778
> in 2010 to 2020 (12%) and 200 in 2020 to 2023 (3.1%), with 118 unknown.

Criterion: the `evaluation_year` column, Table 2's bins (closed left, open right, the last
closed), percentages over rows with a known year, U1 or U2; all nine values within tolerance.

In [ ]:
bins = jecfa.year_bin(df.evaluation_year)
counts = bins.value_counts().reindex(jecfa.YEAR_BINS)
known = int(bins.notna().sum())
res = pd.DataFrame({"count": counts, "share_of_known_pct": (counts / known * 100).round(2),
                    "expected_count": [2357, 3099, 778, 200], "expected_pct": [37, 48, 12, 3.1]})
res["count_within"] = (res["count"] - res.expected_count).abs() <= (res.expected_count * 0.1).round(0)
res["pct_within"] = (res.share_of_known_pct - res.expected_pct).abs() <= res.expected_pct * 0.1 + 1e-9
print(res.to_string())
print(f"Unknown year: {int(df.evaluation_year.isna().sum())} (expected 118); known {known}; "
      f"years outside 1958 to 2023: {int((bins.isna() & df.evaluation_year.notna()).sum())}")
print("Shares over all 6552 rows:", (counts / len(df) * 100).round(2).tolist())

**Result: reproduced.** The four bins hold 2357, 3099, 778 and 200 rows and 118 years are missing,
equal to Table 2; the shares of known years are 36.63, 48.17, 12.09 and 3.11%, all within
tolerance.

## table2-functional-class

> Most substances are flavouring agents, 4635 (72%), followed by food additives 1453 (23%),
> veterinary drugs 277 (4.3%) and food contaminants 77 (1.2%), with 110 unknown.

Criterion: the leading class name of `functional_class`, case-insensitive, empty values unknown;
percentages over rows with a known class; all nine values within tolerance and flavouring agents
the largest class.

In [ ]:
cc = cls.value_counts().reindex(jecfa.CLASSES)
known_c = int(cls.notna().sum())
res = pd.DataFrame({"count": cc, "share_of_known_pct": (cc / known_c * 100).round(3),
                    "expected_count": [4635, 1453, 77, 277], "expected_pct": [72, 23, 1.2, 4.3]})
res["count_within"] = (res["count"] - res.expected_count).abs() <= res.expected_count * 0.1
res["pct_within"] = (res.share_of_known_pct - res.expected_pct).abs() <= res.expected_pct * 0.1 + 1e-9
print(res.to_string())
print(f"Unknown class: {int(cls.isna().sum())} (expected 110); known {known_c}; "
      f"strings matching none of the four names: {int((cls.isna() & df.functional_class.notna()).sum())}")
print("Largest class:", cc.idxmax())

**Result: reproduced.** 4635 flavouring agents, 1453 food additives, 77 food contaminants, 277
veterinary drugs and 110 unknown, equal to Table 2; the shares of known classes are 71.95, 22.56,
1.20 and 4.30%. Flavouring agents are the largest class, and every string starts with one of the
four class names.

## unknowns-name-only

> The records with unknown class or year have only the JECFA name, without chemical names or other
> information.

Criterion: among the rows with unknown functional class, and among those with unknown evaluation
year, the share with no Table 1 field present besides URL and JECFA name; reproduced when it is at
least 90% of each set.

In [ ]:
others = [c for c in df.columns if c not in ("url", "jecfa_name")]
name_only = ~present[others].any(axis=1)
for label, mask in [("unknown functional class", cls.isna()), ("unknown evaluation year", df.evaluation_year.isna())]:
    n, k = int(mask.sum()), int((mask & name_only).sum())
    print(f"{label}: {n} rows, {k} with only URL and JECFA name ({k / n * 100:.1f}%)")
    print("   fields present in the other", n - k, "rows:",
          present.loc[mask & ~name_only, others].sum()[lambda s: s > 0].to_dict())
print("\nUnknown class and unknown year together:", int((cls.isna() & df.evaluation_year.isna()).sum()),
      "; rows with only URL and JECFA name in the whole table:", int(name_only.sum()))

**Result: partially reproduced.** 95 rows carry only a URL and a JECFA name, and all of them have
both an unknown class and an unknown year. They are 86.4% of the 110 rows with an unknown class
and 80.5% of the 118 with an unknown year, below the 90% of the criterion. The other 15 rows with
an unknown class carry a year, a report and identifiers (15 have a JECFA number, 13 a FEMA
number), and the other 23 with an unknown year all carry a functional class, 15 of them chemical
names and 14 synonyms.

## fas-codes

> Almost all 85 FAS codes are referenced, and FAS 40, 42, 44, 46, 48, 50 and 52 together account for
> 50% of the records.

Criterion: FAS codes from the column the code derives; the share of the seven named codes over
(F1) all rows or (F2) rows with a FAS code, within 5 points of 50%; the seven named codes the seven
most frequent for reproduced, partially reproduced when the share holds and they are not. "Almost
all 85" is at least 77 distinct non-empty codes. As for table1-availability, the deposited column
(A) and the column after `add_metadata.R` (B) are both run.

In [ ]:
named = ["40", "42", "44", "46", "48", "50", "52"]
for label, fas in [("A (deposited fas)", df.fas), ("B (after add_metadata.R)", fas_b)]:
    has = jecfa.present(fas)
    vc = fas[has].value_counts()
    in_named = int(fas[has].isin(named).sum())
    f1, f2 = in_named / len(df) * 100, in_named / int(has.sum()) * 100
    top7 = vc.index[:7].tolist()
    print(f"Reading {label}: {int(has.sum())} rows with a FAS code, {vc.size} distinct codes (at least 77: {vc.size >= 77})")
    print(f"   rows in the seven named codes: {in_named}; F1 {f1:.2f}% (within 45 to 55: {abs(f1 - 50) <= 5}), "
          f"F2 {f2:.2f}% (within: {abs(f2 - 50) <= 5})")
    print(f"   seven most frequent: {top7}; the named seven: {set(top7) == set(named)}; "
          f"ranks of the named codes: {[int(vc.index.get_loc(c)) + 1 for c in named]}")
    ranked = pd.DataFrame({"code": vc.index[:12], "rows": vc.values[:12],
                           "cum_share_F2_pct": (vc.cumsum() / has.sum() * 100).round(1).values[:12],
                           "cum_share_F1_pct": (vc.cumsum() / len(df) * 100).round(1).values[:12]})
    print(ranked.to_string(index=False), "\n")

**Result: partially reproduced.** The seven named codes hold 3036 rows, 46.34% of all rows (F1,
within 5 points of 50%) under both readings, and 56.94% (A) or 53.42% (B) of the rows with a FAS
code (F2). They are not the seven most frequent: FAS 59 (351 rows) is more frequent than FAS 52
(286), and under reading B so is FAS 60 (353). The deposited column has 81 distinct values, some
malformed ("77 JECFA 86", "67.29", "617"); after `add_metadata.R` there are 73 codes, fewer than the
77 taken for "almost all 85".

## validation-failures

> In a manual check of 700 random pages, scraped and manually read values disagreed only for Report
> (0.6%), Report sourcelink (0.9%), Tox monograph sourcelink (5.3%) and Specification sourcelink
> (0.2%).

Blocked, `data_missing`. The test needs the manual extraction of the 700 sampled pages, entered in
REDCap at the University of Padova. It is not in the Figshare item (whose two files are the
scraped table, checked above), nor in the paper's GitHub repository at the commit read, and the
paper gives no other place for it. The live portal has changed since May 2024, so the extraction
cannot be repeated on the pages as they were.

## portal-search-count

> A search of the JECFA portal by first character returned 6547 items, in line with the pages
> carrying a JECFA name (6551 less a duplicate and a test page).

Criterion: the portal count (6547) is outside data that is not public in its May 2024 state and is
untestable. For the dataset's part: the rows with a JECFA name present, 6551 within 655, and after
the cleaning of unique-pages within 2 of 6547.

In [ ]:
with_name = int(present.jecfa_name.sum())
print(f"Rows with a JECFA name present: {with_name} (expected 6551, tolerance 655; within: {abs(with_name - 6551) <= 655})")
for r, v in zip(readings.reading, readings.unique_pages):
    print(f"After cleaning, reading {r}: {v}; difference from 6547: {v - 6547} (within 2: {abs(v - 6547) <= 2})")
print("The paper's own numbers: 6551 - 2 = 6549, a difference of 2 from 6547")

**Result: partially reproduced.** All 6552 rows carry a JECFA name, within tolerance of the
paper's 6551, but one more. After the cleaning of unique-pages 6550 pages remain (D2, D3), 3 more
than the portal's 6547, outside the margin of 2; D1 gives 6552. The portal count itself cannot be
tested.

## Summary

| Claim | Outcome |
|---|---|
| data-records-rows (link) | reproduced |
| unique-pages | reproduced |
| table1-availability | reproduced |
| table2-evaluation-year | reproduced |
| table2-functional-class | reproduced |
| unknowns-name-only | partially reproduced |
| fas-codes | partially reproduced |
| validation-failures | blocked (data_missing) |
| portal-search-count | partially reproduced (dataset's part only) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The paper publishes no code for its Table 1 percentages,
Table 2 or the FAS counts, so the measures follow the claims file's criteria. The deposit holds
the FAS column of `process_df.R`; the further step of `add_metadata.R` is run as a second reading.
The manual validation data are not public.

What was found about the dataset.

- The deposit holds the 6552 scraped rows, including the test page (5883) and the stub page of a
  chemical listed twice (6316, whose record is at 6319); the paper's 6549 unique pages are not
  what is deposited, and the paper's statement that each entry is a unique chemical does not hold
  for these two rows.
- JECFA.csv and jecfa.rds hold the same table.
- The item's description calls FAS a "Food Additive Specification identifier"; the column is the
  Food Additives Series number derived from the Tox monograph. It misses FAS monographs whose text
  starts with '">' (mostly FAS 60), holds 752 empty strings beside 468 NA, and some values keep a
  session or page ("77 JECFA 86", "67.29", "617").
- Table 1 describes the COE number as assigned by the Codex Alimentarius Commission, the item's
  description as the Council of Europe number; the numbers occur for flavouring agents almost only,
  consistent with the Council of Europe.
- A few plain CAS numbers fail the check digit, and values of report, Tox monograph and
  specification start with '">', as the Usage Notes say.